# Lab 10: Strojenie hiperparametrów MLP z Optuną
### Biblioteki Python w analizie danych
**Tomasz Rodak**

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rodakt/BPwAD/blob/v2/laby/lab_10.ipynb)

W tym arkuszu łączymy dwa wątki kursu: trening sieci neuronowych w PyTorch (lab 7–9) ze systematycznym strojeniem hiperparametrów Optuną (lab 6, wykład 3). Zbudujemy MLP do regresji na zbiorze California Housing, uruchomimy strojenie hiperparametrów dwukrotnie — najpierw bez prunignu, potem z `MedianPruner`-em raportującym po każdej epoce — i na koniec wytrenujemy model finalny na połączonym zbiorze treningowo-walidacyjnym.

Pruning po epokach to naturalne rozszerzenie tego, co robiliśmy w lab 6. Tam raportowaliśmy średnią accuracy po kolejnych foldach kroswalidacji; tutaj raportujemy val MSE po kolejnych epokach treningu sieci. Mechanizm jest dokładnie ten sam (`trial.report` + `trial.should_prune`), zmienia się tylko, co stanowi "krok" pośredni.


In [ ]:
pip install optuna


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)


## 1. Dane


### 1.1 Wczytanie i podział

Zbiór California Housing pochodzi ze spisu z 1990 roku. Każdy rekord opisuje grupę bloków mieszkalnych (block group) w Kalifornii. Cechy: mediana dochodu, mediana wieku budynków, liczba pokoi i sypialni, populacja, liczba gospodarstw domowych oraz współrzędne geograficzne. Target — mediana ceny domu w setkach tysięcy dolarów.


In [ ]:
data = fetch_california_housing()
X, y = data.data.astype(np.float32), data.target.astype(np.float32)
print(f"Kształt: X={X.shape}, y={y.shape}")
print(f"Cechy: {data.feature_names}")


Dzielimy na zbiory treningowy, walidacyjny i testowy w proporcji 60/20/20:


In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42  # 0.25 * 0.8 = 0.2
)
print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")


**Higiena: zbiór testowy odkładamy na bok.** Używamy go **tylko raz**, w sekcji 7, do oceny modelu finalnego. W szczególności scaler dopasowujemy na zbiorze treningowym (sekcja 1.2), a w trakcie strojenia hiperparametrów kierujemy się wyłącznie błędem na zbiorze walidacyjnym. Nie korzystamy z kroswalidacji, gdyż taka optymalizacja mogłaby się nam nie zmieścić w czasie zajęć.

### 1.2 Standaryzacja i tensory

Cechy mają wyraźnie różne skale (mediana dochodu vs współrzędne geograficzne), więc standaryzacja jest konieczna.

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)


Konwersja na tensory `float32`. Zbiory walidacyjny i testowy zostawiamy jako surowe tensory (do ewaluacji bez `DataLoader`-a):


In [ ]:
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
y_train_t = torch.tensor(y_train,   dtype=torch.float32)
X_val_t   = torch.tensor(X_val_s,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,     dtype=torch.float32)
X_test_t  = torch.tensor(X_test_s,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,    dtype=torch.float32)

train_dataset = TensorDataset(X_train_t, y_train_t)


Zwróć uwagę: `DataLoader` na zbiorze treningowym **jeszcze nie powstaje**. `batch_size` będzie hiperparametrem strojonym przez Optunę, więc loader trzeba budować **wewnątrz** funkcji celu. To istotne; gdyby loader powstał teraz, każda próba i tak korzystałaby z tej samej wartości `batch_size` i strojenie tego hiperparametru byłoby pozorne.


### 1.3 Krótka eksploracja

Histogram targetu i korelacja cech z targetem:


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(y_train, bins=50)
ax1.set_xlabel("Cena (×100 000 USD)")
ax1.set_title("Rozkład targetu (train)")

corrs = [np.corrcoef(X_train[:, i], y_train)[0, 1] for i in range(X_train.shape[1])]
ax2.barh(data.feature_names, corrs)
ax2.set_xlabel("Korelacja Pearsona z ceną")
ax2.set_title("Korelacja cech z targetem")
plt.tight_layout()


Zwróć uwagę na "obcięcie" w histogramie — wartości > 5 zostały sklejone w jedną wartość przez urzędników spisu. To realna właściwość danych, którą żaden model nie nauczy się dobrze przewidywać. Zobaczymy ją w sekcji 7 jako poziomy klaster punktów na wykresie predykcji.

Najsilniejszą korelację z ceną ma mediana dochodu — co nie powinno dziwić.


## 2. Baseline'y

Zanim wpuścimy Optunę, ustalmy poziom odniesienia. Dwa modele: Ridge (rozwiązanie zamknięte) i ręcznie zaprojektowany MLP z "rozsądnymi" hiperparametrami. Liczby, które wyjdą na zbiorze walidacyjnym, będziemy potem chcieli pobić.


### 2.1 Ridge

Regresja liniowa z $\ell_2$-regularyzacją. Trenujemy na połączonym `train+val`? **Nie** — to byłoby nieuczciwe porównanie z modelami z sekcji 3–5, które będą widziały tylko `train`. Ridge ma trenować na tym samym zbiorze co MLP, więc trening na samym `train` i ewaluacja na `val`:


In [ ]:
ridge = Ridge(alpha=1.0).fit(X_train_s, y_train)
val_mse_ridge = mean_squared_error(y_val, ridge.predict(X_val_s))
print(f"Ridge val MSE: {val_mse_ridge:.4f}")


### 2.2 Funkcja pomocnicza `train_and_evaluate`

Pętla treningowa będzie nam potrzebna **cztery razy**: dla baseline'u MLP poniżej, dla funkcji celu Optuny bez pruningu (sekcja 3), z pruningiem (sekcja 5) oraz dla modelu finalnego (sekcja 7). Zamiast pisać ją cztery razy, zdefiniujmy raz funkcję pomocniczą:


In [ ]:
from pyexpat import model


def train_and_evaluate(model, train_loader, X_val, y_val,
                       optimizer, num_epochs, trial=None):
    """Trenuje model, po każdej epoce raportuje val MSE.

    Jeśli trial jest podany, raportuje wartość pośrednią do Optuny
    i sprawdza, czy próba powinna zostać odrzucona (pruning).
    """
    criterion = nn.MSELoss()
    val_mse = float("inf")
    for epoch in range(num_epochs):
        # TODO 1: trening jednej epoki w kanonicznej pętli
        # zero_grad → forward → loss → backward → step
        # Pamiętaj o squeeze(-1) na wyjściu modelu (kształt (B, 1) → (B,))
        model.train()
        for xb, yb in train_loader:
            ...


        # TODO 2: ewaluacja na val w model.eval() + torch.no_grad()
        # Oblicz val_mse jako pojedynczą liczbę (item())
        model.eval()
        with torch.no_grad():
            ...

        # Raportowanie do Optuny — jeśli trial podany
        if trial is not None:
            trial.report(val_mse, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    return val_mse


*Uwaga o `squeeze(-1)`.* `nn.Linear(in, 1)` zwraca tensor o kształcie `(B, 1)`, a etykiety mają kształt `(B,)`. Bez `squeeze` `nn.MSELoss` wykona broadcast do `(B, B)` i zwróci nonsensowną wartość bez żadnego błędu. Spotykaliśmy to w lab 9 sekcja 4.

*Uwaga o `trial=None`.* Domyślna wartość pozwala używać tej samej funkcji w kontekście, w którym Optuny nie ma w ogóle (baseline w 2.3, model finalny w sekcji 7). Gdy `trial` jest podany — funkcja raportuje val MSE do Optuny po każdej epoce i pozwala prunerowi przerwać trening.


### 2.3 Ręczny MLP

Dwie warstwy ukryte po 64 i 32 neurony, ReLU, Adam z domyślnym `lr=1e-3`, `batch_size=256`, 20 epok:


In [ ]:
torch.manual_seed(0)
model_baseline = nn.Sequential(
    nn.Linear(8, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1),
)
loader_baseline = DataLoader(train_dataset, batch_size=256, shuffle=True)
optimizer = torch.optim.Adam(model_baseline.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
val_mse_mlp = train_and_evaluate(
    model_baseline, loader_baseline, X_val_t, y_val_t,
    optimizer, num_epochs=20
)
print(f"MLP baseline val MSE: {val_mse_mlp:.4f}")


Te dwie liczby — Ridge i MLP baseline — to nasz **punkt odniesienia**. Chcemy, żeby Optuna znalazła konfigurację MLP, która je pobije (i to wyraźnie, nie o trzecie miejsce po przecinku).


## 3. Funkcja celu dla Optuny

Pora na sedno labu — zbudujemy funkcję celu, w której Optuna sama będzie projektować architekturę MLP i stroić jego hiperparametry treningu. To pierwszy kontekst, w którym strojona jest **architektura sieci** (liczba warstw, szerokość) razem z hiperparametrami treningu (`lr`, `weight_decay`, `batch_size`).


### 3.1 Przestrzeń przeszukiwania

Pięć hiperparametrów:

| Hiperparametr | Typ | Zakres |
|---|---|---|
| `n_layers` | int | 1–3 |
| `hidden_size` | categorical | {32, 64, 128} |
| `lr` | float, log | $10^{-4}$–$10^{-2}$ |
| `weight_decay` | float, log | $10^{-6}$–$10^{-2}$ |
| `batch_size` | categorical | {128, 256} |

`weight_decay` to $\ell_2$-regularyzacja wbudowana w optymalizator — nasz odpowiednik `alpha` z Ridge'a, ale strojony przez Optunę (zamiast ustawiany ręcznie). Ustawiana jest jako parametr `Adam(weight_decay=...)`.

Optymalizator zostawiamy ustawiony na sztywno na Adama — z lab 9 wiemy, że Adam jest wyraźnie mniej wrażliwy na dobór `lr` niż SGD i traktujemy go jako "domyślny" wybór dla sieci neuronowych. Strojenie SGD obok Adama wymagałoby warunkowych przestrzeni przeszukiwania (różne zakresy `lr` dla SGD i Adama) — zostawiamy jako zadanie 8.4.

### 3.2 Define-by-run dla architektury

Sieć nie jest stała — buduje się w funkcji celu zależnie od wylosowanego `n_layers` i `hidden_size`. To technika *define-by-run* z lab 6 (sekcja 3.1, gdzie wybór klasyfikatora wpływał na resztę przestrzeni).


In [ ]:
def build_mlp(n_layers, hidden_size, in_features=8):
    """Buduje MLP z n_layers warstwami ukrytymi po hidden_size neuronów."""
    layers = []
    prev = in_features
    for _ in range(n_layers):
        layers += [nn.Linear(prev, hidden_size), nn.ReLU()]
        prev = hidden_size
    layers.append(nn.Linear(prev, 1))
    return nn.Sequential(*layers)


### 3.3 Funkcja celu — wersja podstawowa


In [1]:
NUM_EPOCHS = 20

def objective(trial):
    # Reprodukowalność: każdy trial deterministyczny względem swojego numeru
    torch.manual_seed(trial.number)

    # TODO 1: pięć suggest_* zgodnie z tabelą w 3.1
    n_layers     = trial.suggest_int(...)
    hidden_size  = trial.suggest_categorical(...)
    lr           = trial.suggest_float(..., log=True)
    weight_decay = trial.suggest_float(..., log=True)
    batch_size   = trial.suggest_categorical(...)

    # TODO 2: zbuduj model i optymalizator
    # Optymalizator: Adam z lr i weight_decay
    model = ...
    optimizer = ...

    # DataLoader buduje się TUTAJ, bo batch_size jest hiperparametrem
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # TODO 3: wytrenuj i zwróć val MSE
    # Bez pruningu — trial=None
    return train_and_evaluate(...)

Dwa szczegóły, na które warto zwrócić uwagę:

- **`torch.manual_seed(trial.number)`**. Bez ustawiania seeda model inicjalizuje się losowo przy każdym uruchomieniu funkcji celu, co psuje porównania między trialami (a po wprowadzeniu prunera w sekcji 5 — utrudnia mu pracę). Każdy trial dostaje swój własny, deterministyczny seed na podstawie numeru — to daje powtarzalność i jednocześnie zachowuje różnorodność między trialami.
- **`DataLoader` budowany w środku.** Przy `shuffle=True` musi być tworzony świeży na każdy trial, bo `batch_size` zmienia się między trialami. `train_dataset` (`TensorDataset`) możemy reużywać — on jest niezależny od `batch_size`.


## 4. Optymalizacja bez pruningu

Najpierw study bez prunera. Zobaczymy, ile czasu zajmuje 10 pełnych triali (każdy 20 epok), i to będzie nasz baseline czasowy do porównania z pruningiem w sekcji 5.


### 4.1 Storage i study

Tak jak w lab 6, używamy SQLite jako storage'u, żeby móc oglądać wyniki w Optuna Dashboard. Dla świeżego startu przy każdym uruchomieniu notebooka kasujemy istniejący plik bazy:


In [ ]:
import os
DB_PATH = "lab10.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
storage = f"sqlite:///{DB_PATH}"

Tworzymy study i uruchamiamy:


In [ ]:
study_plain = optuna.create_study(
    study_name="california_plain",
    storage=storage,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    load_if_exists=True,
)

t0 = time.time()
study_plain.optimize(objective, n_trials=10)
time_plain = time.time() - t0

print(f"Czas: {time_plain:.1f} s")
print(f"Best val MSE: {study_plain.best_value:.4f}")
print(f"Best params: {study_plain.best_params}")


*Liczba triali na zajęciach.* 10 triali to za mało, żeby strojenie miało dużą moc statystyczną — to liczba dobrana pod budżet czasowy zajęć (90 min na napisanie i przerobienie całości). W domu warto powtórzyć z `n_trials=50`, żeby zobaczyć, jak różnica względem baseline'u się stabilizuje. Domyślny TPE potrzebuje 10 startup-trials zanim zacznie kierować przeszukiwaniem, więc przy `n_trials=10` praktycznie cała optymalizacja jest losowa. Czujny czytelnik zauważy, że sensowniej byłoby zmniejszyć `n_startup_trials` w `TPESampler` — tak, ale to detal nieistotny dla mechaniki labu.


### 4.2 Wizualizacje

Trzy standardowe wykresy z `optuna.visualization`:


In [ ]:
optuna.visualization.plot_optimization_history(study_plain).show()
optuna.visualization.plot_parallel_coordinate(study_plain).show()
optuna.visualization.plot_param_importances(study_plain).show()


W historii optymalizacji powinieneś zobaczyć krzywe schodzące w dół. W parallel coordinate — czy rysują się wyraźne wzorce (np. czy najlepsze próby grupują się w określonym rejonie `lr`). Importances pokażą, który hiperparametr ma największy wpływ na val MSE.


### 4.3 Dashboard

W osobnej komórce (lub w terminalu, jeśli pracujesz lokalnie):


In [ ]:
# !pip install optuna-dashboard -q
# !optuna-dashboard sqlite:///lab10.db &


Dashboard otwórz w przeglądarce na `http://localhost:8080`. W Colabie potrzebny jest tunel (np. `localtunnel`); na zajęciach możesz tę część pominąć i wrócić do niej w domu — wszystkie informacje są dostępne też przez `optuna.visualization`.


## 5. Pruning po epokach

Teraz drugi przebieg z prunerem. Modyfikacja minimalna: w funkcji celu przekazujemy `trial` do `train_and_evaluate` (a ta po każdej epoce raportuje val MSE i sprawdza `should_prune`).


### 5.1 Funkcja celu z pruningiem


In [ ]:
def objective_pruned(trial):
    torch.manual_seed(trial.number)

    # TODO 1: pięć suggest_* zgodnie z tabelą w 3.1
    n_layers     = trial.suggest_int(...)
    hidden_size  = trial.suggest_categorical(...)
    lr           = trial.suggest_float(..., log=True)
    weight_decay = trial.suggest_float(..., log=True)
    batch_size   = trial.suggest_categorical(...)


    model = build_mlp(n_layers, hidden_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    return train_and_evaluate(
        model, train_loader, X_val_t, y_val_t,
        optimizer, NUM_EPOCHS, trial=trial    # ← jedyna zmiana
    )


Praktycznie identyczne z `objective` z 3.3 — z jedną drobną zmianą `trial=trial` w wywołaniu `train_and_evaluate`. Tam, w środku funkcji pomocniczej, dzieje się reszta:

```python
if trial is not None:
    trial.report(val_mse, step=epoch)
    if trial.should_prune():
        raise optuna.TrialPruned()
```


### 5.2 Pruner i drugie study

`MedianPruner` — ten sam co w lab 6, ale o innym znaczeniu kroku:


In [ ]:
study_pruned = optuna.create_study(
    study_name="california_pruned",
    storage=storage,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=3),
    load_if_exists=True,
)

t0 = time.time()
study_pruned.optimize(objective_pruned, n_trials=20)
time_pruned = time.time() - t0


*Parametry prunera.* `n_startup_trials=3` znaczy, że pierwsze 3 próby wykonują się bez prunignu — pruner potrzebuje historii referencyjnej. `n_warmup_steps=3` znaczy, że w każdym pojedynczym trialu pierwsze 3 epoki nie są prunowane — model musi mieć szansę się rozkręcić. To 15% z 20 epok; wartość w bezpiecznej strefie. (W zadaniu 8.2 sprawdzimy, jaką oszczędność czasu daje agresywniejszy pruning.)


### 5.3 Statystyki pruningu


In [ ]:
pruned   = [t for t in study_pruned.trials if t.state == optuna.trial.TrialState.PRUNED]
complete = [t for t in study_pruned.trials if t.state == optuna.trial.TrialState.COMPLETE]

print(f"Czas: {time_pruned:.1f} s")
print(f"Zakończone: {len(complete)}, odrzucone: {len(pruned)}")
print(f"Best val MSE: {study_pruned.best_value:.4f}")


W dashboardzie (jeśli go uruchomiłeś) zwróć uwagę na wykres *Intermediate Values* — odrzucone próby mają krótkie krzywe (urywają się tam, gdzie pruner przerwał trening); zakończone mają pełne krzywe długości 20.


## 6. Porównanie

Zbierzmy wszystko w jednej tabeli:


In [ ]:
results = pd.DataFrame([
    {"model": "Ridge",            "val_MSE": val_mse_ridge,        "complete": "—", "pruned": "—", "czas (s)": "—"},
    {"model": "MLP baseline",     "val_MSE": val_mse_mlp,          "complete": "—", "pruned": "—", "czas (s)": "—"},
    {"model": "Optuna bez prun.", "val_MSE": study_plain.best_value,
                                  "complete": len(study_plain.trials), "pruned": 0,
                                  "czas (s)": f"{time_plain:.1f}"},
    {"model": "Optuna + pruning", "val_MSE": study_pruned.best_value,
                                  "complete": len(complete), "pruned": len(pruned),
                                  "czas (s)": f"{time_pruned:.1f}"},
])
print(results.to_string(index=False))


Co powinno być widać:

- **Optuna bez pruningu** poprawia val MSE w stosunku do MLP baseline, ale efekt może być umiarkowany przy 10 trialach.
- **Optuna z pruningiem** w podobnym (lub mniejszym) czasie wykonuje **dwa razy więcej triali** (20 vs 10), z których część została odrzucona po kilku epokach. Best val MSE powinien być co najmniej tak dobry jak przy 10 pełnych trialach, zwykle lepszy.
- Liczba odrzuconych triali będzie zależna od tego, jak rozproszone są wartości pośrednie — przy 20 trialach typowo można oczekiwać 5–10 odrzuconych.

Dydaktyczny wniosek: pruning daje **więcej eksploracji w tym samym budżecie czasowym**. Im dłuższy trening jednego trialu, tym większy zysk.


## 7. Finalny model i ewaluacja na teście

Ostatni krok: zbudować model z najlepszymi hiperparametrami, wytrenować go na **połączonym** zbiorze `train + val`, a wynik podać na zbiorze testowym. To dotąd nieużywany zbiór — pierwsza i ostatnia jego rola w całym labie.


### 7.1 Refit scalera i tensorów

Skoro zmienia się zbiór treningowy (teraz to `train+val`), scaler trzeba dopasować ponownie. **Test set nadal w izolacji** — nigdy nie wchodzi do `fit`:


In [ ]:
scaler_final = StandardScaler().fit(X_trainval)
X_trainval_s = scaler_final.transform(X_trainval)
X_test_s_f   = scaler_final.transform(X_test)

X_trainval_t = torch.tensor(X_trainval_s, dtype=torch.float32)
y_trainval_t = torch.tensor(y_trainval,   dtype=torch.float32)
X_test_t_f   = torch.tensor(X_test_s_f,   dtype=torch.float32)

trainval_dataset = TensorDataset(X_trainval_t, y_trainval_t)


### 7.2 Trening modelu finalnego

Wybieramy najlepsze hiperparametry — zwykle z lepszego z dwóch study, ale dla jasności weźmiemy `study_pruned`:


In [ ]:
best = study_pruned.best_params
print("Najlepsze hiperparametry:", best)

torch.manual_seed(0)
model_final = build_mlp(best["n_layers"], best["hidden_size"])
optimizer_final = torch.optim.Adam(
    model_final.parameters(), lr=best["lr"], weight_decay=best["weight_decay"]
)
loader_final = DataLoader(trainval_dataset, batch_size=best["batch_size"], shuffle=True)

# Możesz przetrenować dłużej skoro to ostatni trening — np. 30 epok
train_and_evaluate(
    model_final, loader_final, X_test_t_f, y_test_t,
    optimizer_final, num_epochs=30
)


**Uwaga.** Parametry `X_val`, `y_val` w wywołaniu `train_and_evaluate` służą tu wyłącznie do raportowania val MSE w trakcie treningu — nie używamy tej wartości do żadnej decyzji (early stopping, wybór epoki). Liczbę epok ustalamy z góry.


### 7.3 Ostateczna ocena


In [ ]:
model_final.eval()
with torch.no_grad():
    y_pred_test = model_final(X_test_t_f).squeeze(-1).numpy()

test_mse = mean_squared_error(y_test, y_pred_test)
test_rmse = np.sqrt(test_mse)
print(f"Test MSE:  {test_mse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}  (≈ {test_rmse * 100_000:.0f} USD)")


RMSE w jednostkach naturalnych — czyli przeciętny błąd przewidywanej ceny w dolarach. Liczba rzędu kilkudziesięciu tysięcy USD jest dla California Housing typowa.


### 7.4 Wykres prawdziwa cena vs predykcja


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_test, alpha=0.2, s=8)
lims = [0, 5.5]
ax.plot(lims, lims, 'r--', label='y = x')
ax.set_xlabel("Prawdziwa cena (×100k USD)")
ax.set_ylabel("Predykcja (×100k USD)")
ax.set_xlim(lims); ax.set_ylim(lims)
ax.legend(); ax.set_aspect('equal')
ax.set_title("Predykcja modelu finalnego na zbiorze testowym")


Co powinno być widać:

- Punkty zgrupowane wokół prostej $y = x$ — model działa.
- **Pionowa kolumna punktów przy $y = 5$** — to obcięcie ceny, które zauważyliśmy w 1.3. Prawdziwa cena w tych przypadkach była ≥ 5, ale została skrócona do 5; model uczy się właściwej ceny i wypluwa różne wartości, podczas gdy prawda jest sztywno na 5. To przypadek, w którym model robi dobrą rzecz, ale zostaje za to "ukarany" przez RMSE.
- Asymetria: błędy bywają większe dla droższych domów (prawa strona wykresu).


## 8. Zadania dodatkowe


### 8.1 Funkcja aktywacji jako hiperparametr

Dorzuć do przestrzeni `activation = trial.suggest_categorical("activation", ["relu", "tanh", "gelu"])` i zmodyfikuj `build_mlp`, by uwzględniał wybór. Powtórz strojenie z pruningiem (`n_trials=20`). Czy któraś aktywacja systematycznie wygrywa? Przejrzyj `plot_param_importances` — czy nowy hiperparametr ma istotny wpływ?


### 8.2 Agresywniejszy pruning

Powtórz sekcję 5 z `MedianPruner(n_startup_trials=3, n_warmup_steps=1)`. Co się zmienia: liczba ukończonych vs odrzuconych triali, czas, najlepsze val MSE? Czy zdarza się, że pruner przerwał trial, który po kilku epokach by się rozkręcił? (Wskazówka: porównaj rozkład wartości w 4. i 5. epoce dla zakończonych vs odrzuconych prób — `study.trials_dataframe()` z parametrem `attrs=("intermediate_values",)`).


### 8.3 TPE vs Random

Zrób trzecie study z `RandomSampler(seed=42)` zamiast TPE, na tej samej przestrzeni i z tym samym pruningiem co w sekcji 5, `n_trials=20`. Porównaj krzywe `plot_optimization_history` obu study na jednym wykresie. TPE powinien zacząć wyraźnie wyprzedzać Random gdzieś po startup-trialach (~10 trial). Przy 20 trialach efekt może być jeszcze niewyraźny — uruchomienie z `n_trials=50` pokaże go lepiej.


### 8.4 SGD jako alternatywa dla Adama

Dorzuć do przestrzeni hiperparametr `optimizer ∈ {"adam", "sgd"}`. Pamiętaj, że SGD wymaga innego zakresu `lr` niż Adam (typowo $10^{-3}$–$10^{-1}$). Najczystsze rozwiązanie to **warunkowa przestrzeń** — zakres `lr` zależy od wybranego optymalizatora:


In [ ]:
optimizer_name = trial.suggest_categorical("optimizer", ["adam", "sgd"])
if optimizer_name == "adam":
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
else:
    lr = trial.suggest_float("lr", 1e-3, 1e-1, log=True)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)


Czy SGD przy odpowiednim `lr` może dorównać Adamowi? Zobacz w parallel coordinate, jakie kombinacje wyłaniają się jako najlepsze.
